# Spring 2024 6.8200 Computational Sensorimotor Learning Assignment 6

In this assignment, we will implement model-based control algorithms for Cartpole swing-up.

You will need to **answer the bolded questions** and **fill in the missing code snippets** (marked by **TODO**).

There are **210** total points to be had in this PSET, plus 10 bonus points for filling out the survey.  `ctrl-f` for "pts" to ensure you don't miss questions.

In [ ]:
!sudo apt-get update > /dev/null 2>&1
!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install gym > /dev/null 2>&1
!pip install git+https://github.com/taochenshh/easyrl.git > /dev/null 2>&1
!pip install stable-baselines3 > /dev/null 2>&1
!pip install "shimmy>=0.2.1" > /dev/null 2>&1

In [ ]:
%matplotlib inline

import gym
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches
from matplotlib.animation import FuncAnimation
import matplotlib.animation as animation
from matplotlib.axes import Axes
from matplotlib import rc
import random
from torch import nn
from gym.envs.registration import registry, register
from tqdm.notebook import tqdm

np.bool = np.bool_

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

rc('animation', html='jshtml')

# Environment

For this assignment, our task will be to swing-up and balance Cartpole (note that this is different from the standard cartpole gym env, where the stick starts in the upwards orientation).  As a backbone to our gym environment, we provide the class `CartpoleDynamics` which implements the ground truth cartpole dynamics in Pytorch (for speed).

Note that in the controls literature, $q$ is commnly referred to as the state and $u$ as the control or input or action.

Note that our observation space in this environment is of size 4 and the action space is of size 1.

*Cartpole plotting and animation code inspired by [Shunichi09/PythonLinearNonLinearControl](https://github.com/Shunichi09/PythonLinearNonlinearControl)*

In [ ]:
class CartpoleDynamics:
    def __init__(self,
                 timestep=0.02,
                 m_p=0.5,
                 m_c=0.5,
                 l=0.6,
                 g=-9.81,
                 u_range=15):
        """
        Initializes the Cartpole Dynamics model with given parameters.

        Parameters:
        - timestep (float): The time step for the simulation.
        - m_p (float): Mass of the pole.
        - m_c (float): Mass of the cart.
        - l (float): Length of the pole.
        - g (float): Acceleration due to gravity. Negative values indicate direction.
        - u_range (float): Range of the control input.
        """

        self.m_p  = m_p
        self.m_c  = m_c
        self.l    = l
        self.g    = -g
        self.dt   = timestep

        self.u_range = u_range

        self.u_lb = torch.tensor([-1]).float()
        self.u_ub = torch.tensor([1]).float()
        self.q_shape = 4
        self.u_shape = 1

    def _qdotdot(self, q, u):
        """
        Calculates the acceleration of both cart and pole as a function of the current state and control input.

        Parameters:
        - q (torch.Tensor): The current state of the system, [x, theta, xdot, thetadot].
        - u (torch.Tensor): The current control input.

        Returns:
        - torch.Tensor: Accelerations [x_dotdot, theta_dotdot] of the cart and the pole.
        """
        x, theta, xdot, thetadot = q.T

        if len(u.shape) == 2:
            u = torch.flatten(u)

        x_dotdot = (
            u + self.m_p * torch.sin(theta) * (
                self.l * torch.pow(thetadot,2) + self.g * torch.cos(theta)
            )
        ) / (self.m_c + self.m_p * torch.sin(theta)**2)

        theta_dotdot = (
            -u*torch.cos(theta) -
            self.m_p * self.l * torch.pow(thetadot,2) * torch.cos(theta) * torch.sin(theta) -
            (self.m_c + self.m_p) * self.g * torch.sin(theta)
        ) / (self.l * (self.m_c + self.m_p * torch.sin(theta)**2))

        return torch.stack((x_dotdot, theta_dotdot), dim=-1)

    def _euler_int(self, q, qdotdot):
        """
        Performs Euler integration to calculate the new state given the current state and accelerations.

        Parameters:
        - q (torch.Tensor): The current state of the system, [x, theta, xdot, thetadot].
        - qdotdot (torch.Tensor): The accelerations [x_dotdot, theta_dotdot] of the cart and the pole.

        Returns:
        - torch.Tensor: The new state of the system after a single time step.
        """

        qdot_new = q[...,2:] + qdotdot * self.dt
        q_new = q[...,:2] + self.dt * qdot_new

        return torch.cat((q_new, qdot_new), dim=-1)

    def step(self, q, u):
        """
        Performs a single step of simulation given the current state and control input.

        Parameters:
        - q (torch.Tensor or np.ndarray): The current state of the system.
        - u (torch.Tensor or np.ndarray): The current control input.

        Returns:
        - torch.Tensor: The new state of the system after the step.
        """

        # Check for numpy array
        if isinstance(q, np.ndarray):
            q = torch.from_numpy(q)
        if isinstance(u, np.ndarray):
            u = torch.from_numpy(u)

        scaled_u = u * float(self.u_range)

        # Check for shape issues
        if len(q.shape) == 2:
            q_dotdot = self._qdotdot(q, scaled_u)
        elif len(q.shape) == 1:
            q_dotdot = self._qdotdot(q.reshape(1,-1), scaled_u)
        else:
            raise RuntimeError('Invalid q shape')

        new_q = self._euler_int(q, q_dotdot)

        if len(q.shape) == 1:
            new_q = new_q[0]

        return new_q

    # given q [bs, q_shape] and u [bs, t, u_shape] run the trajectories
    def run_batch_of_trajectories(self, q, u):
        """
        Simulates a batch of trajectories given initial states and control inputs over time.

        Parameters:
        - q (torch.Tensor): Initial states for each trajectory in the batch.
        - u (torch.Tensor): Control inputs for each trajectory over time.

        Returns:
        - torch.Tensor: The states of the system at each time step for each trajectory.
        """
        qs = [q]

        for t in range(u.shape[1]):
            qs.append(self.step(qs[-1], u[:,t]))

        return torch.stack(qs, dim=1)

    # given q [bs, t, q_shape] and u [bs, t, u_shape] calculate the rewards
    def reward(self, q, u):
        """
        Calculates the reward for given states and control inputs.

        Parameters:
        - q (torch.Tensor or np.ndarray): States of the system.
        - u (torch.Tensor or np.ndarray): Control inputs applied.

        Returns:
        - torch.Tensor: The calculated rewards for the states and inputs.
        """

        if isinstance(q, np.ndarray):
            q = torch.from_numpy(q)
        if isinstance(u, np.ndarray):
            u = torch.from_numpy(u)

        angle_term = 0.5*(1-torch.cos(q[...,1]))
        pos_term = -0.5*torch.pow(q[...,0],2)
        ctrl_cost = -0.001*(u**2).sum(dim=-1)

        return angle_term + pos_term + ctrl_cost

class CartpoleGym(gym.Env):
    def __init__(self, timestep_limit=200):
        """
        Initializes the Cartpole environment with a specified time step limit.

        Parameters:
        - timestep_limit (int): The maximum number of timesteps for each episode.

        Sets up the dynamics model and initializes the simulation state.
        """
        self.dynamics = CartpoleDynamics()

        self.timestep_limit = timestep_limit
        self.reset()

    def reset(self):
        """
        Resets the environment to the initial state.

        Returns:
        - np.ndarray: The initial state of the environment.
        """

        self.q_sim = np.zeros(4)
        self.timesteps = 0

        self.traj = [self.get_observation()]

        return self.traj[-1]

    def get_observation(self):
        """
        Retrieves the current state of the environment.

        Returns:
        - np.ndarray: The current state of the simulation.
        """

        return self.q_sim

    def step(self, action):
        """
        Executes one time step within the environment using the given action.

        Parameters:
        - action (np.ndarray): The action to apply for this timestep.

        Returns:
        - Tuple[np.ndarray, float, bool, dict]: A tuple containing the new state, the reward received,
          a boolean indicating whether the episode is done, and an info dictionary.
        """
        action = np.clip(action, self.action_space.low, self.action_space.high)[0]

        new_q = self.dynamics.step(
            self.q_sim, action
        )

        if not isinstance(action, torch.Tensor):
            action = torch.tensor(action)

        reward = self.dynamics.reward(
            new_q, action
        ).numpy()

        self.q_sim = new_q.numpy()
        done = self.is_done()

        self.timesteps += 1

        self.traj.append(self.q_sim)

        return self.q_sim, reward, done, {}

    def is_done(self):
        """
        Checks if the episode has finished based on the timestep limit.

        Returns:
        - bool: True if the episode is finished, False otherwise.
        """
        # Kill trial when too much time has passed
        if self.timesteps >= self.timestep_limit:
            return True

        return False

    def plot_func(self, to_plot, i=None):
        """
        Plots the current state of the cartpole system for visualization.

        Parameters:
        - to_plot (matplotlib.axes.Axes or dict): Axes for plotting or a dictionary of plot elements to update.
        - i (int, optional): The index of the current state in the trajectory to plot.
        """
        def _square(center_x, center_y, shape, angle):
            trans_points = np.array([
                [shape[0], shape[1]],
                [-shape[0], shape[1]],
                [-shape[0], -shape[1]],
                [shape[0], -shape[1]],
                [shape[0], shape[1]]
            ]) @ np.array([
                [np.cos(angle), np.sin(angle)],
                [-np.sin(angle), np.cos(angle)]
            ]) + np.array([center_x, center_y])

            return trans_points[:, 0], trans_points[:, 1]

        if isinstance(to_plot, Axes):
            imgs = dict(
                cart=to_plot.plot([], [], c="k")[0],
                pole=to_plot.plot([], [], c="k", linewidth=5)[0],
                center=to_plot.plot([], [], marker="o", c="k",
                                          markersize=10)[0]
            )

            x_width = max(1,max(np.abs(t[0]) for t in self.traj) * 1.3)

            # centerline
            to_plot.plot(np.linspace(-x_width, x_width, num=50), np.zeros(50),
                         c="k", linestyle="dashed")

            # set axis
            to_plot.set_xlim([-x_width, x_width])
            to_plot.set_ylim([-self.dynamics.l*1.2, self.dynamics.l*1.2])

            return imgs

        curr_x = self.traj[i]

        cart_size = (0.15, 0.1)

        cart_x, cart_y = _square(curr_x[0], 0.,
                                cart_size, 0.)

        pole_x = np.array([curr_x[0], curr_x[0] + self.dynamics.l
                           * np.cos(curr_x[1]-np.pi/2)])
        pole_y = np.array([0., self.dynamics.l
                           * np.sin(curr_x[1]-np.pi/2)])

        to_plot["cart"].set_data(cart_x, cart_y)
        to_plot["pole"].set_data(pole_x, pole_y)
        to_plot["center"].set_data(self.traj[i][0], 0.)

    def render(self, mode="human"):
        """
        Renders the current state of the environment using a matplotlib animation.

        This function creates a matplotlib figure and uses the plot_func method to update the figure with the current
        state of the cartpole system at each timestep. The animation is created with the FuncAnimation class and is
        configured to play at a specified frame rate.

        Parameters:
        - mode (str): The mode for rendering. Currently, only "human" mode is supported, which displays the animation
          on screen.

        Returns:
        - matplotlib.animation.FuncAnimation: The animation object that can be displayed in a Jupyter notebook or
          saved to file.
        """
        self.anim_fig = plt.figure()

        self.axis = self.anim_fig.add_subplot(111)
        self.axis.set_aspect('equal', adjustable='box')

        imgs = self.plot_func(self.axis)
        _update_img = lambda i: self.plot_func(imgs, i)

        Writer = animation.writers['ffmpeg']
        writer = Writer(fps=15, metadata=dict(artist='Me'), bitrate=1800)

        ani = FuncAnimation(
            self.anim_fig, _update_img, interval=self.dynamics.dt*1000,
            frames=len(self.traj)-1
        )

        plt.close()

        return ani

    @property
    def action_space(self):
        """
        Defines the action space of the environment using a Box space from OpenAI Gym.

        The action space is defined based on the lower and upper bounds for the control input specified in the
        dynamics model. This allows for a continuous range of actions that can be applied to the cartpole system.

        Returns:
        - gym.spaces.Box: The action space as a Box object, with low and high bounds derived from the dynamics model's
          control input bounds.
        """
        return gym.spaces.Box(low=self.dynamics.u_lb.numpy(), high=self.dynamics.u_ub.numpy())

    @property
    def observation_space(self):
        """
        Defines the observation space of the environment using a Box space from OpenAI Gym.

        The observation space is defined with no bounds on the values, representing the position and velocity of the
        cart and the angle and angular velocity of the pole. This space allows for any real-valued vector of
        positions and velocities to be a valid observation in the environment.

        Returns:
        - gym.spaces.Box: The observation space as a Box object, with low and high bounds set to negative and
          positive infinity, respectively, for each dimension of the state vector.
        """
        return gym.spaces.Box(
            low= np.array([-np.inf, -np.inf, -np.inf, -np.inf]),
            high=np.array([np.inf,   np.inf,  np.inf,  np.inf])
        )

env_name = 'CartpoleSwingUp-v0'
if env_name in registry.env_specs:
    del registry.env_specs[env_name]
register(
    id=env_name,
    entry_point=f'{__name__}:CartpoleGym',
)


As a demonstration, let's try to plot a random policy on this environment.

In [ ]:
env = gym.make('CartpoleSwingUp-v0')

q = env.reset()
done = False
while not done:
    q, r, done, _ = env.step(env.action_space.sample())

env.render()

# Q1. Model Predictive Control

Model-based RL works by decomposing the policy into two components:

1. A model of the Markov Decision Process (MDP), learned from data
2. A planner, that given the learned MDP, can produce optimal action estimates.

Let's begin by implementing step 2: the planner.  As we're in the world of continuous control, we will implement a *Model Predictive Controller*: at every timestep we will solve for an optimal trajectory, than take the first action.

One simple MPC method is Cross Entropy Method (CEM)(Section 8.1.1 of lecture notes), as used in [Probabilistic Ensembles with Trajectory Sampling (PETS)](https://proceedings.neurips.cc/paper_files/paper/2018/file/3de568f8597b94bda53149c7d7f5958c-Paper.pdf).  The idea here is to continuously optimize a rolling Gaussian input trajectory segment.  At each timestep, advance the trajectory segment by one, generate a population of new input segments by sampling from the last input segment, and evaluate each trajectory.  Then, select the top `es_elites` trajectories from that population with the highest reward, fit the input trajectory segment to their distribution, and repeat for `es_generations` generations.  If the input trajectory variance reaches below a threshold `es_epsilon`, exit early.

Implement a CEM MPC controller, using the given parameters defined in `__init__` and both `self.model.run_batch_of_trajectories` and `self.model.reward` to evaluate trajectories.

Note that `run_batch_of_trajectories` takes as input a batch of initial states $(bs, u\_size)$ and a trajectory of actions $(bs, t, q\_size)$ and returns a batch of trajectories $(bs, t + 1)$. Each trajectory in the batch is generated by sequentially applying each action in the trajectory to updated states (starting with the initial inputted state). `reward` takes a sequence of states $(bs, t, q\_size)$ and a sequence of actions $(bs, t, u\_size)$ applied to get to that state and returns a reward $(bs, t)$, consisting of a term based off the position, angle, and action magnitude.

**(50 pts)**

In [1]:
class MPC:
    def __init__(self,
        model,
        horizon        = 25,
        es_epsilon     = 0.001,
        es_alpha       = 0.1,
        es_generations = 5,
        es_popsize     = 200,
        es_elites      = 40
    ):
        """
        Initializes the Model Predictive Control (MPC) with an evolutionary strategy (ES) optimizer.

        Parameters:
        - model: The dynamics model to be used for predicting future states.
        - horizon (int): The planning horizon for the MPC.
        - es_epsilon (float): The variance threshold for termination of the ES optimization loop.
        - es_alpha (float): The rolling average coefficient for updating the solution mean and variance.
        - es_generations (int): The number of generations for the ES optimizer to run.
        - es_popsize (int): The population size for each generation in the ES optimizer.
        - es_elites (int): The number of elite samples to use for updating the solution distribution.
        """

        self.model          = model

        self.horizon        = horizon        # planning horizon
        self.es_epsilon     = es_epsilon     # variance threshold
        self.es_alpha       = es_alpha       # new distribution rolling average coefficient
        self.es_generations = es_generations # num generations for ES optimizer
        self.es_popsize     = es_popsize     # popsize for ES optimizer
        self.es_elites      = es_elites      # num of elites from which to resample
        self.vars = []
        self.reset()

    def reset(self):
        """
        Resets the MPC optimizer by initializing the solution mean and variance based on the action bounds
        provided by the model.
        """
        # Initialize action trajectory distribution
        self.sol_mean = ((self.model.u_ub + self.model.u_lb) / 2).expand(self.horizon,-1)
        self.sol_var = ((self.model.u_ub - self.model.u_lb) / 16).expand(self.horizon,-1)

        self.timestep = 0


    def action(self, q):
        """
        Computes an action for the given state using the MPC with ES optimization.

        Parameters:
        - q: The current state of the system.

        Returns:
        - The computed action for the state.

        The method iteratively refines a distribution over actions by sampling trajectories, evaluating them
        using the model, and selecting the top-performing actions to update the distribution.
        """
        # Remove last taken action and add 0 to end of buffer
        self.sol_mean = torch.cat([
            self.sol_mean[1:],
            torch.zeros(self.model.u_shape).reshape(1,-1)
        ])

        # Generate standard diagonal normal distribution from which we sample trajectory noise
        u_dist = torch.distributions.normal.Normal(
            loc=torch.zeros_like(self.sol_mean),
            scale=torch.ones_like(self.sol_var)
        )

        var = self.sol_var
        mean = self.sol_mean
        for n in range(self.es_generations):
            # Terminate if variance drops below threshold
            if torch.max(var) < self.es_epsilon:
                print(f'var below threshold! exiting {n}')
                break

            lb_dist = mean - self.model.u_lb
            ub_dist = self.model.u_ub - mean

            # Clipped form of variance to avoid sampling extreme actions. Use this
            # for sampling, however use 'var' when updating the variance.
            constrained_var = torch.min(
                torch.min((lb_dist / 2)**2, (ub_dist / 2)**2), var
            )


            ### TODO: perform one ES trajectory optimization step, by
            ### 1. sampling a trajectory
            ### 2. evaluating the fitness of that trajectory on the model
            ### 3. re-fitting (updating) mean and variance for the top N elites of the model
            ### (50 pts in  total)

            # Sample input trajectories to be evaluated (10 pts)

            # Evaluate each trajectory (10 pts)

            # Evaluate reward for each population member (10 pts)

            # Resample mean and variance from top performers (10 pts)

            # calculate elite reward (10 pts)

            ### ENDTODO
        self.sol_mean = mean # update sol_mean (we are not updating sol_var)
        self.timestep += 1
        return self.sol_mean[0]

Now, let's evaluate your MPC controller on the ground truth cartpole dynamics `CartpoleDynamics`.  The cartpole should swing up on the first (and only) epoch; there's no learning here, as the dynamics model is known.

In [ ]:
counts = []
for i in range(1):
    mpc = MPC(CartpoleDynamics())
    e = gym.make('CartpoleSwingUp-v0')

    q, r, done = e.reset(), 0, False

    count = 0
    while not done:
        q, reward, done, _ = e.step(mpc.action(q))
        r += reward
        count += 1
    counts.append(r)

print(counts)
print(np.mean(counts))
print(np.std(counts))

print('Got reward:', r)
e.render()


**Question**: Try varying the time horizon for MPC planning.  What behaviors do you see? Visualize the plot (horizon x reward) and answer based on that (10 pts; 5pts for answer, 5 pts for implementation)

**Answer**:

In [ ]:
counts = []
horizons = [5,10,25,50,100]

###TODO Run various Horizons

### ENDTODO
plt.plot(horizons, counts)




**Question**: Try varying the number of random seeds for MPC.  What behaviors do you see? Visualize bar plot for reward (counts) and answer based on that. Are they have similar performance? (10 pts; 5pts for answer, 5 pts for implementation)



**Answer**:

In [ ]:
def set_random_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

In [ ]:
counts = []
seeds = [0,1,2,3,4]

### TODO

#### ENDTODO

# visaulize histogram
plt.bar(seeds, counts)


# Q2. Learned Dynamics Model

Now that we've debugged our MPC module, let's implement a learned model, so we no longer need the ground truth dynamics.  Our goal here is to learn some model $\hat f(q_t, u_t) = \hat q_{t+1}$, rather than implementing it analytically.

You can think about following changes:
- For angles, consider replacing the angle in your feature space with `sin` and `cos` of the angle. $\theta = (cos\theta,sin\theta)$. This is a common form of feature input engineering for data that has cyclical structure. It is [known](https://www.researchgate.net/publication/326542245_Advanced_machine_learning_techniques_for_building_performance_simulation_a_comparative_analysis#pf7) (section 3.3 and Table 4) to increase model performance.
- Normalize $u$ based on its range.
- Consider learning the model $\hat f$ as $\hat f(q_t, u_t) = \hat g(q_t, u_t) + q_t$, rather than predicting new states from scratch (i.e., learn some residual instead)

**(50 points)**

In [ ]:
class LearnedCartpoleDynamics:
    def __init__(self, seed, u_range=15):
        """
        Initializes a learned dynamics model for the cartpole system with specified control input range.

        Parameters:
        - seed (int): The seed for random number generators to ensure reproducibility.
        - u_range (float): The range of the control input.
        """
        set_random_seed(seed)
        self.u_range = u_range

        self.u_lb = torch.tensor([-u_range]).float()
        self.u_ub = torch.tensor([u_range]).float()
        self.u_shape = 1

        input_size = 6 #0 # TODO

        # Input state and control into model predictor
        self.model = nn.Sequential(
            nn.Linear(input_size, 48),
            nn.ReLU(),
            nn.Linear(48, 48),
            nn.ReLU(),
            nn.Linear(48, 4),
            nn.Tanh()
        )

        self.optim = torch.optim.Adam(self.model.parameters())
        self.loss = nn.MSELoss()

    def step(self, q, u):
        """
        Predicts the next state of the system given the current state and control input using the learned model.

        Parameters:
        - q (torch.Tensor): The current state of the system.
        - u (torch.Tensor): The current control input.

        Returns:
        - torch.Tensor: The predicted next state of the system.
        """
        ### TODO: implement forward dynamics call of f(q, u) -> qprime


        ### ENDTODO

        raise NotImplementedError()

    # given q [n, q_shape] and u [n, t] run the trajectories
    def run_batch_of_trajectories(self, q, u):
        """
        Simulates a batch of trajectories given initial states and a sequence of control inputs using the learned dynamics.

        Parameters:
        - q (torch.Tensor): Initial states for each trajectory in the batch.
        - u (torch.Tensor): Control inputs for each trajectory over time.

        Returns:
        - torch.Tensor: The states of the system at each time step for each trajectory.
        """

        qs = [q]

        for t in range(u.shape[1]):
            qs.append(self.step(qs[-1], u[:,t]))

        return torch.stack(qs, dim=1)

    def train(self, q_t_traj, q_tplusone_traj, u_traj):
        """
        Trains the learned dynamics model on a dataset of state transitions and control inputs.

        Parameters:
        - q_t_traj (np.ndarray): Array of current states.
        - q_tplusone_traj (np.ndarray): Array of next states corresponding to each current state.
        - u_traj (np.ndarray): Array of control inputs applied to transition from current to next states.

        Optimizes the model parameters to minimize the prediction error between the predicted and actual next states.
        """

        batch_size = 16
        num_batches = 1024

        ### TODO: train your forward dynamics model by minimizing the mean squared error between
        ###       predicted future states (given a current state and control) and actual future states
        ###
        ### note: we've already defined the optimizer in self.optim and the loss function in self.loss


       ### ENDTODO

    def reward(self, q, u):
        """
        Calculates the reward for a given state and control input based on the cartpole dynamics.

        Parameters:
        - q (torch.Tensor or np.ndarray): States of the system.
        - u (torch.Tensor or np.ndarray): Control inputs applied.

        Returns:
        - torch.Tensor: The calculated rewards for the states and inputs based on the angle and position of the
          cartpole and the cost of control inputs.
        """
        if isinstance(q, np.ndarray):
            q = torch.from_numpy(q)
        if isinstance(u, np.ndarray):
            u = torch.from_numpy(u)

        angle_term = 0.5*(1-torch.cos(q[...,1]))
        pos_term = -0.5*torch.pow(q[...,0],2)
        ctrl_cost = -0.001*(u**2).sum(dim=-1)

        return angle_term + pos_term + ctrl_cost

Let's see how we do!  Run the below cell to run a trial where you
1. roll-out an episode using the MPC controller on the learned dynamics model
2. use the newly gathered experience to update the dynamics model
3. goto step 1.

In [ ]:
# Expected runtime ~ 1 min. in T4
nsteps = 0
import time
epochs_to_solve = []
for _ in range(1):
    dynamics_model = LearnedCartpoleDynamics(seed=0)
    mpc = MPC(dynamics_model)
    e = CartpoleGym()

    start = time.time()
    all_q, all_q_prime, all_u = None, None, None
    for epoch in range(100): # It should end before 10 epochs.
        q_traj = [e.reset()]
        u_traj = []

        r = 0
        while True:
            u_traj.append(mpc.action(q_traj[-1])[0].item())
            q, reward, done, _ = e.step(u_traj[-1])
            r += reward
            q_traj.append(q)
            nsteps += 1

            if done:
                break

        q_traj = np.array(q_traj)
        u_traj = np.array(u_traj)

        print(f'[Epoch {epoch}] Got reward {r}')
        q, q_prime, u = q_traj[:-1], q_traj[1:], u_traj

        if all_q is None:
            all_q, all_q_prime, all_u = q, q_prime, u
        else:
            all_q = np.concatenate((all_q, q))
            all_q_prime = np.concatenate((all_q_prime, q_prime))
            all_u = np.concatenate((all_u, u))

        dynamics_model.train(all_q, all_q_prime, all_u)

        if r > 90:
            epochs_to_solve.append(epoch)
            break
    end = time.time()

    e.render()

    print(f"{end - start} seconds")

print(epochs_to_solve)
print(np.mean(epochs_to_solve))
print(np.std(epochs_to_solve))
print(nsteps)

**Question:** What is the number of steps needed to stop for the model-based algorithm? (10 pts)

**Answer:**

Not bad! Now let's train a standard ppo agent on the same task and see how it compares to our model based approach.

In [ ]:
# ~5 minutes on T4
num_experiments = 1

import time
start = time.time()

set_random_seed(0)

print("Training model... Typically takes <5 min")

model = PPO("MlpPolicy", env, verbose=0)
model.learn(total_timesteps=100000)
model.save("ppo_cartpole")

n_steps = 0

for _ in range(num_experiments):
  obs = env.reset()
  ppo_epochs = 100
  epochs_to_solve = []

  for epoch in tqdm(range(ppo_epochs)):
    obs = env.reset()
    epoch_reward = 0
    while True:
        action, _states = model.predict(obs)
        obs, reward, done, info = env.step(action)
        n_steps += 1
        epoch_reward += reward
        if done:
          break

    print(f'[Epoch {epoch}] Got reward {epoch_reward}')

    if epoch_reward > 90:
        epochs_to_solve.append(epoch)
        break

end = time.time()

print(f"{end - start} seconds")
print(epochs_to_solve)
print(np.mean(epochs_to_solve))
print(np.std(epochs_to_solve))
print(n_steps)
env.render()

**Question:** Compare the number of steps needed for training of our model-based approach to previously explored methods such as PPO. (10 pts)

**Answer:**

# Q3. Robust MPC by Model Ensembling

Now, instead of using just one single learned forward dynamics model, let's utilize an **ensemble** of forward dynamics models as done in [PETS](https://arxiv.org/pdf/1805.12114.pdf). We will have multiple dynamics model networks, as defined in the last section, that are randomly initialized. (Algorithm description in Section 8.1.1- Algorithm 8 in lecture notes).

Like before, for each of these models we will generate trajectories given the same initial state and sampled control trajectory. We will rank all of our models by their **best** performing elite trajectories, and then use the elites of the **worst** performing model to update our sampling each generation. This conservative approach should lead to more stable behavior.

**(50 points)**

In [ ]:
class EnsembleMPC:
    def __init__(self,
        models,
        horizon        = 25,
        es_epsilon     = 0.001,
        es_alpha       = 0.1,
        es_generations = 5,
        es_popsize     = 200,
        es_elites      = 40
    ):
        """
        Initializes an Ensemble Model Predictive Control (MPC) strategy that uses multiple models for decision making.

        Parameters:
        - models (list[CartpoleDynamics]): A list of dynamics models that are part of the ensemble.
        - horizon (int): The planning horizon for the MPC.
        - es_epsilon (float): The variance threshold for early stopping in the optimization process.
        - es_alpha (float): The learning rate for updating the mean and variance of the solution distribution.
        - es_generations (int): The maximum number of generations for the evolutionary strategy (ES) optimization.
        - es_popsize (int): The population size used in each generation of the ES optimization.
        - es_elites (int): The number of elite candidates to consider when updating the solution distribution.
        """

        self.models          = models
        self.num_in_ensemble = len(self.models)

        self.horizon        = horizon        # planning horizon
        self.es_epsilon     = es_epsilon     # variance threshold
        self.es_alpha       = es_alpha       # new distribution rolling average coefficient
        self.es_generations = es_generations # num generations for ES optimizer
        self.es_popsize     = es_popsize     # popsize for ES optimizer
        self.es_elites      = es_elites      # num of elites from which to resample

        self.reset()

    def reset(self):
        """
        Resets the internal state of the EnsembleMPC, including the solution mean and variance.
        """

        # Initialize action trajectory distribution
        self.sol_mean = ((self.models[0].u_lb + self.models[0].u_ub) / 2).expand(self.horizon,-1)
        # solution variacne is fixed in each episode.
        self.sol_var = ((self.models[0].u_ub - self.models[0].u_lb) / 16).expand(self.horizon,-1)

        self.timestep = 0

    def action(self, q):
        """
        Computes the optimal action to take for a given state using the ensemble of models.

        Parameters:
        - q (array-like): The current state of the system from which to compute the optimal action.

        Returns:
        - torch.Tensor: The computed action that is expected to be optimal according to the ensemble MPC strategy.
        """
        # Remove last taken action and add 0 to end of buffer
        self.sol_mean = torch.cat([
            self.sol_mean[1:],
            torch.zeros(self.models[0].u_shape).reshape(1,-1)
        ])

        # Generate standard diagonal normal distribution from which we sample trajectory noise
        u_dist = torch.distributions.normal.Normal(
            loc=torch.zeros_like(self.sol_mean),
            scale=torch.ones_like(self.sol_var)
        )

        var = self.sol_var # we will not update self.sol_var.
        mean = self.sol_mean
        for n in range(self.es_generations):
            # Terminate if variance drops below threshold
            if torch.max(var) < self.es_epsilon:
                print(f'var below threshold! exiting {n}')
                break

            lb_dist = self.sol_mean - self.models[0].u_lb
            ub_dist = self.models[0].u_ub - self.sol_mean
            constrained_var = torch.min(
                torch.min((lb_dist / 2)**2, (ub_dist / 2)**2), var
            )

            ### TODO: perform one ES trajectory optimization step, by
            ### 1. sampling a trajectory from each model using the same initial state and
            ###    sampled control trajectory
            ### 2. evaluating the fitness of that trajectory on all models.
            ###    Hint: Use model.run_batch_of_trajectories
            ### 3. calculating the average reward of the top N elites from each model
            ### 4. re-fitting self.sol_mean for the top N elites of the model with the lowest
            ###    average elite reward (i.e., taking a conservative action)
            ### (50 pts)

            # Sample input trajectories to be evaluated

            # Evaluate each trajectory

            # Evaluate reward for each population member

            # calculate elite rewards

            # Resample mean and variance from worst performer

            ### ENDTODO
            self.sol_mean = mean # only update mean not variance.

        self.timestep += 1
        return self.sol_mean[0]

In [ ]:
# Expected runtime ~ 7 min. in T4
epochs_to_solve = []
nsteps = 0
for _ in range(1):
    num_in_ensemble = 6
    dynamics_models = [LearnedCartpoleDynamics(seed=idx) for idx in range(num_in_ensemble)]
    mpc_ensemble = EnsembleMPC(dynamics_models)
    e = CartpoleGym()

    start = time.time()
    all_q, all_q_prime, all_u = None, None, None

    for epoch in range(100): # It should end at much earlier epoch.
        q_traj = [e.reset()]
        u_traj = []

        r = 0
        while True:
            u_traj.append(mpc_ensemble.action(q_traj[-1])[0].item())
            q, reward, done, _ = e.step(u_traj[-1])
            nsteps += 1
            r += reward
            q_traj.append(q)

            if done:
                break

        q_traj = np.array(q_traj)
        u_traj = np.array(u_traj)

        print(f'[Epoch {epoch}] Got reward {r}')
        q, q_prime, u = q_traj[:-1], q_traj[1:], u_traj

        if all_q is None:
            all_q, all_q_prime, all_u = q, q_prime, u
        else:
            all_q = np.concatenate((all_q, q))
            all_q_prime = np.concatenate((all_q_prime, q_prime))
            all_u = np.concatenate((all_u, u))

        for idx in range(num_in_ensemble):
            dynamics_models[idx].train(all_q, all_q_prime, all_u)

        end = time.time()

        if r > 90:
            epochs_to_solve.append(epoch)
            break
    end = time.time()

    e.render()

    print(f"{end - start} seconds")
print(epochs_to_solve)
print(np.mean(epochs_to_solve))
print(np.std(epochs_to_solve))
print(nsteps)

**Question:** What is the number of steps needed for training for the ensemble model-based algorithm? (10 pts)

**Answer:**

**Question:** How does the ensemble method compare to the method without ensembling in terms of learning speed (how many epoch is needed, how much time it needed to finish)? (10 pts)

**Answer:**


# Feedback Survey (optional)

Please enter the bonus code you get after filling out the [anonymous assignment survey](https://forms.gle/KNwV6doNq1PCiTZw8). (10 pts).

**Bonus code**: